# Basics of mobile robotics – Project report

### Student names (SCIPER):
* Bastien Marconato (330256)
* Selma Benhassine  (300148)
* Mischa Mez        (310752)
* Davide Lisi       (375197)

# Vision

The computer vision component utilizes functions from the OpenCV library. Functions like `VideoCapture()` and `cap.read()` are employed to access the camera frame. Object detection is achieved through the use of masks, particularly color detection in this case. The obstacles are green, the robot is white, and the goal is red. To facilitate effective masking, a background of a different color (blue) is chosen when the robot is on a white background.

The mask creation process involves converting the image/frame from the BGR color space to the HSV (Hue, Saturation, Value) color space. Subsequently, masking is performed by adjusting the values of the HSV image to isolate the desired color. A Gaussian blurring filter is applied to improve the mask by eliminating noise. A Canny edge filter is then used for precise line detection of the object. The contours of the objects are obtained using the `cv2.findContours()` function, providing a list of points representing the object's contour.

For localization of the robot and the goal, information from the contour method, specifically `cv2.minAreaRect()`, is utilized. This function creates a box around the object, providing parameters such as the center, width, and height of the object in pixels, essential for localization.

In the Global Navigation and Kalman filter components, the position information of the robot and the goal needs to be in millimeters rather than pixels. To convert pixel measurements to millimeters, a ratio is calculated using known dimensions in millimeters. If W_pixel and W_mm are the width in pixels and millimeters, respectively, the ratio is determined by:

$$\text{Ratio} = \frac{W_{\text{mm}}}{W_{\text{pixel}}} \\$$


This ratio is then used to convert any pixel measurement to millimeters with the formula:

$$\ Measurement_{mm} = Pixel \ Measurement \times Ratio \\$$

Lastly, we can mention the `findAngle()` function, which is used to find the orientation of the robot. This is needed for the Kalman filter to predict the position of the robot. The method involves placing two squares, one big square, and a smaller square, and then calculating the angle between them. The angle $\theta$ can be determined using the following equation:

$$\theta = \arctan\left(\frac{\Delta y}{\Delta x}\right)$$

To ensure the angle is in radians and falls between $-\pi$ and $\pi$, the `math.atan2()` function from is used.

Here we can look at an example of code to find an object detection. It follows always the same principle. We use a mask with `cv2.inRange()`, which is a function that returns a white binary image where the colors are detected and 0 otherwise. Then we do a blurring of the image to try to get rid of the little noise part in the image, we use a Canny edge filter. We already so in class and in the exercice session the Canny edge filter, and how it can be use to detect line (contours in our case)

In [ ]:
mask = cv2.inRange(img, WHITE_LOWER, WHITE_UPPER)
blurred_img = cv2.GaussianBlur(mask, (7, 7), 0)    #(7,7) is a good value for accounting for noise
edges = cv2.Canny(blurred_img, 100, 255)           #(100, 255) set experimentaly and worked
contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE) #RETR_EXTERNAL retrieves only the extreme outer contours

Here a quick review on the mathematics of the Canny Edge filter from OpenCV (https://docs.opencv.org/4.x/da/d22/tutorial_py_canny.html):

![Canny Edge Principle](Computer_vision/Canny_edge.JPG)

In order to then decide what is and what is not an edge, we do thresholding using to values a minVal and a Max Val. Here is the comment on OpenCV documentation about it:

![Thresholding](Computer_vision/minMax.JPG)

We just show another part of the code, which is about correctly detection our objects. In case of the angle calculation, we need to differentiate the 3 red colored papers. One is our goal and the other two are our elements to calculate the angle of the robot. One is bigger then the other, in order to differentiate them, otherways we couldn't do the calculation. To detect each one of them, we use their areas by using the openCV function `cv2.contourArea()`. Here is a bit of code showing the procedure

In [ ]:
results = []
for contour in contours:
    area = cv2.contourArea(contour)

    if 30000 < area < 60000:    #To only localise the ROBOT
        rect = cv2.minAreaRect(contour) #fits the minimum rectangle, in order to get a dynamic rectangle, that also rotate when the robot rotate
        box = cv2.boxPoints(rect)
        box = np.int0(box) #we create a box to show on the video the robot
        cv2.drawContours(video, [box], 0, (0, 0, 255), 3)

        center = (int(rect[0][0]), int(rect[0][1]))
        angle = int(rect[2])
        width, height = (int(rect[1][0]), int(rect[1][1]))
        #cv2.putText(video, f"{area}", (center[0], center[1]+50), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,0), 2)

        results.append((center, -angle, (width, height), box)) 

Here we also juste show the code for the conversion form pixel to MM:

In [ ]:
def positionToMM(results, results_goal): #convert a pixel location into MM
    ratio= PIXEL2MM(results_goal)
    x_cm = (results[0][0][0])*ratio
    y_cm = (results[0][0][1])*ratio
    return x_cm, y_cm

def PIXEL2MM(results_goal): # takes the goal width as information, because it is taken once and doesn't change afterwards
    if not results_goal:
        print("No object detected.")
        return None, None, None
    
    width = results_goal[0][2][0]
    ratio = GOAL_WIDTH_MM/width

    return ratio

## Global navigation
For global navigation, we have developed a function that takes the coordinates of obstacle vertices, as well as the positions of the start and goal, as input. The function outputs the coordinates of points representing the shortest path from the start to the goal. This is accomplished by computing the visibility graph of the obstacles after resizing and merging (refer to sections below for the definition of resized and merged). Subsequently, a Dijkstra weighted algorithm is applied between the nodes of the graph in order to identify the coordinates (in pixels for now), that the robot should pass by.\
We calculate the visibility graph to determine all potential linear trajectories. This graph retains information about "visible" vertices for each vertex of a polygon situated on a 2D map (in this instance). For every "visibility line" (a line connecting two vertices without intersecting any polygon edges), the visibility graph records its length, serving as the weight for our Dijkstra Algorithm. However, prior to computation, it is essential to resize and merge the polygons.

#### Resize the obstacles
To prevent the robot from colliding with obstacles, our priority is to ensure that the center of the robot maintains a significant distance from the vertices of the obstacles. We achieved this by enlarging the polygons provided as input to the visibility graph calculation. Refer to the next image for a clearer visualization.\
![Visibility Graph example](Figures\Global_Nav\visibility_graph.jpg){Width=25%}\
To resize an obstacle, we computed the position of a point inside it, representing the coordinates of all its vertices. The following equations depict the calculation of this point (referred to as "incenter" in our code, although it may not necessarily be an incenter as not all obstacles might be inscribable).\
Let $(x_i, y_i)$ represent the coordinates of the vertices. Calculate lengths of sides opposite each vertex:
$$ s_i = \text{distance}((x_i, y_i), (x_{(i + 1) \bmod \text{num\_points}}, y_{(i + 1) \bmod \text{num\_points}})) $$
Calculate incenter coordinates:
$$ x_{\text{incenter}} = \frac{\sum_{i=0}^{\text{nu
m\_points}-1} s_i \cdot x_i}{\sum_{i=0}^{\text{num\_points}-1} s_i} $$
$$ y_{\text{incenter}} = \frac{\sum_{i=0}^{\text{num\_points}-1} s_i \cdot y_i}{\sum_{i=0}^{\text{num\_points}-1} s_i} $$
For the coordinates of each vertex, we determine the resized version by employing the following equations.
Calculate differences in coordinates:
$$ \Delta x = x_{\text{{incenter}}} - x_i \quad \text{and} \quad \Delta y = y_{\text{{incenter}}} - y_i .$$
Calculate the distance between the point and incenter:
$$ d = \text{{distance}}(\text{{point}}, \text{{incenter}}) .$$
Calculate resized coordinates:
$$ x_{resized} = x_i - D_{robot} \cdot \left(\frac{\Delta x}{d}\right) \quad \text{and} \quad y_{resized}  = y_i - D_{robot} \cdot \left(\frac{\Delta y}{d}\right) .$$
The parameter $D_{robot}$ is contingent on the pixel dimensions of the robot. It is recommended to be at least half the width of the robot, but for precautionary measures, we have opted to set it equal to the entire width of the robot. The selection of this parameter is grounded in a trade-off between ensuring security to avoid obstacles and the risk of potentially overlooking simpler paths. When enlarging the size of obstacles, certain paths may appear obstructed during the computation of the visibility graph, even if they are not.\
The function in the code responsible for performing all of these tasks is named in the code below resize_polys(polys_list, robot_d).

#### Merge the obstacles
Resizing the obstacles may lead to instances of overlap, preventing the robot from passing through. In such cases, certain vertices need to be excluded from the visibility graph computation. This task is accomplished through the utilization of a Python library called Shapely.\
The function in the code responsible for performing all of these tasks is named merge_overlapping_polygons(polygons).\
![Merging](Figures\Global_Nav\merging.jpg){Width=25%}\

### Visibility graph
Once all the polygons have been resized and merged we can compute the Visibility Graph. For this task we used a python package called Pyvisgraph.\
Source: Pyvisgraph @ MIT, [GitHub: Pyvisgraph](https://github.com/TaipanRex/pyvisgraph)\
The visibility graph is determined by identifying the visible vertices of the obstacles (refer to the Figure "Visibility Graph example"). Two vertices are considered visible only if the line between them does not intersect with any edge of the obstacles. To determine if two lines intersect, two conditions must be checked for every pair of non-adjacent vertices (general case of non-collinear vertices):\
* (p1, q1, p2) and (p1, q1, q2) must have different orientations.
* (p2, q2, p1) and (p2, q2, q1) must have different orientations.\
![Line crossing check](Figures\Global_Nav\line_cross_check.png){Width=25%} \

If the line between two non-adjacent vertices intersects with at least one edge of the polygons in the space, there is no visibility line between them.\
Source: Geeksforgeeks @ URL, [Check if two Line segments intersect](https://www.geeksforgeeks.org/check-if-two-given-line-segments-intersect/?ref=lbp)

### Dijkstra algorithm
Dijkstra's Algorithm is a fundamental method used in graph theory and computer science to find the shortest path between two nodes in a graph. This algorithm efficiently determines the shortest distance from a starting node to all other nodes in a weighted graph.\
1. Initialize the distance $D(0)$ of the starting node to 0 and all other nodes $D(i>0)$ to infinity.
2. Initialize the shortest path for the starting node to the starting node itself, and the shortest path of all other nodes to an empty list.
3. Initialize the set of nodes yet to process, $P$, to the starting node.
4. For each node $i$ in $P$, for each node $j$ visible from node $i$:
   - Compute a tentative distance of node $j$ as $dtent = D(i) + d(i, j)$.
   - If $dtent$ is smaller than $D(j)$, assign $D(j) = dtent$, set the shortest path for node $j$ to the shortest path for node $i$ augmented with node $j$, and add node $j$ to $P$.
   - Once all nodes $j$ have been investigated, remove node $i$ from $P$.
5. Repeat the last step until $P$ is empty.\

Source: Calerga @ URL, [Mobile robot navigation: Dijkstra method](https://calerga.ch/projects/epfl/mobots/18/nav-visibility.html)

### Global Navigation results
As an outcome of global navigation, we obtain a list of points in pixel coordinates that represent the positions the robot should traverse to reach the goal via the shortest available path in a specific configuration.\
In the code, tha main function for global navigation is global_navigation(polys_list, robot_dimension, start, goal).\
In the next image, we can see that the path planning works even though the recognition of the edges of the polygons isn't perfect. This is because we only need the vertices position, which are really well found by the Vision module.\
![Global Navigation example](Figures\Global_Nav\shortest_path.png){Width=25%}\

In [ ]:
#GLOBAL NAVIGATION MODULE
#This code section contains only the instresting functions: 
#resize_polys(polys_list, robot_d) - 
#merge_overlapping_polygons(polygons)
#global_navigation(polys_list, robot_dimension, start, goal)


# Function to calculate distance between two points
def distance(point1, point2):
    return math.sqrt((point1[0] - point2[0])**2 + (point1[1] - point2[1])**2)

# Function to calculate "incenter" of an obstacle
def incenter(vertices):
    x_coords, y_coords = zip(*vertices)
    num_points = len(vertices)

    # Calculate lengths of sides opposite each vertex
    side_lengths = [distance(vertices[i], vertices[(i + 1) % num_points]) for i in range(num_points)]

    # Calculate incenter coordinates
    incenter_x = sum(side_lengths[i] * x_coords[i] for i in range(num_points)) / sum(side_lengths)
    incenter_y = sum(side_lengths[i] * y_coords[i] for i in range(num_points)) / sum(side_lengths)

    return incenter_x, incenter_y

# Function to resize polygons based on robot dimensions
def resize_polys(polys_list, robot_d):
    incenters = []
    polys_resized = []
    for i in range(0, len(polys_list)):
        polygon = polys_list[i]
        inc_x, inc_y = incenter(polygon)
        incenters.append((inc_x, inc_y))
        polys_resized.append([])
        for j in range(0, len(polygon)):
            point = polygon[j]
            dx = incenters[i][0] - point[0]
            dy = incenters[i][1] - point[1]
            d = distance(point, incenters[i])
            xp = point[0] - robot_d * (dx/d)
            yp = point[1] - robot_d * (dy/d)
            polys_resized[i].append((xp, yp))
    print(f"----------------\nRESIZED POLYGONS:\n{polys_resized}\n")
    return polys_resized, incenters

def merge_overlapping_polygons(polygons):
    # Convert the list of vertex coordinates to Shapely Polygon objects
    shapely_polygons = [Polygon(vertices) for vertices in polygons]

    # Initialize an empty list to store the merged polygons
    merged_polygons = []

    # Iterate through each polygon and check for overlaps
    for i in range(len(shapely_polygons)):
        current_polygon = shapely_polygons[i]
        merged = False

        # Check for overlaps with previously merged polygons
        for j in range(len(merged_polygons)):
            if current_polygon.intersects(merged_polygons[j]):
                # If there is an overlap, merge the polygons
                current_polygon = unary_union([current_polygon, merged_polygons[j]])
                merged_polygons[j] = current_polygon
                merged = True
                break

        # If no overlap was found, add the polygon to the list of merged polygons
        if not merged:
            merged_polygons.append(current_polygon)

    # Convert the final merged polygons back to the list of vertices
    result = [list(p.exterior.coords) for p in merged_polygons]

    return result


def global_navigation(polys_list, robot_dimension, start, goal):
    #PATH CALCULATION
    #Input
    #polys_list = list of the coordinates in  pixels of the vertices of each object
    #robot_dimension = dimension of the robot in pixels
    #Output
    #shortest_list: list of points the robot needs to pass by
    #path_data: it stores the vertices'coordinates and the distance of each segment of the path
    #polys_list_resized_merged: list of the vartices of the combined obstacles
    #incenters_list: list of the incenters of the obstacles
    polys_list_resized, incenters_list = resize_polys(polys_list, robot_dimension+15) #Considering the size of the robot
    polys_list_resized_merged = merge_overlapping_polygons(polys_list_resized) #Considering overlapping resized polygons
    polys = convert_polys_list(polys_list_resized_merged)

    # Build visibility graph
    g = vg.VisGraph()
    g.build(polys)

    # Find the shortest path
    shortest = g.shortest_path(vg.Point(start[0], start[1]), vg.Point(goal[0], goal[1]))
    
    #Converting the results to correct format
    shortest_list = convert_path_to_list(shortest, start, goal)
    path_data = analyze_path(shortest_list)

    # Record the end time
    end_time = time.time()
    # Calculate and print the elapsed time
    elapsed_time = end_time - start_time
    print(f"Elapsed time: {elapsed_time} seconds\nAvaible frequency: {round(1/elapsed_time)} Hz\n")
    return path_data, polys_list_resized_merged, incenters_list

## Pose estimation


### Kinematic model

$$\begin{cases}
\dot{x} = \cos{\theta} \cdot v \\
\dot{y} = \sin{\theta} \cdot v \\
\dot{\theta} = w
\end{cases}$$

$$\begin{cases}
v = \frac{v_r + v_l}{2} \\
w = \frac{v_r - v_l}{2l}
\end{cases}$$

where :
* $\theta$ is the orientation of the robot
* $v$ is the tangential speed [m/s]
* $w$ is the angular speed  [rad/s]
* $v_r$ is the speed of the right wheel (on the ground) [m/s]
* $v_l$ is the speed of the left wheel (on the ground) [m/s]
* $l$ is half of the distance between the two wheels [m]

Source: Autonomous Mobile Robots @ ETHZ, [Exercice 2: Mobile Robot Kinematics and Control](https://ethz.ch/content/dam/ethz/special-interest/mavt/robotics-n-intelligent-systems/asl-dam/documents/lectures/autonomous_mobile_robots/spring-2020/Exercise2/amr_exercise_2_slides.pdf)

Therefore
$$\begin{cases}
\dot{x} = \cos{\theta} \cdot \frac{v_r + v_l}{2} \\
\dot{y} = \sin{\theta} \cdot \frac{v_r + v_l}{2} \\
\dot{\theta} = \frac{v_r - v_l}{2l}
\end{cases}$$



### Extended Kalman Filter
The wheel speeds are considered as part of the system state and we use the encoders to estimate these values. Therefore, our non-linear system can be described by the following continuous-time transition function:
$$ \dot{\mathbf{x}} = \begin{bmatrix}
\dot{x} \\
\dot{y} \\
\dot{\theta} \\
\dot{v_r} \\
\dot{v_l}
\end{bmatrix}
= f(x, y, \theta, v_r, v_l) = \frac{1}{2} \begin{bmatrix}
\cos{\theta} \cdot (v_r + v_l) \\
\sin{\theta} \cdot (v_r + v_l) \\
\frac{v_r - v_l}{l} \\
0 \\
0
\end{bmatrix} + \mathbf{w}$$

where $\mathbf{w} \sim \mathcal{N}\bigl(\mathbf{0},\mathbf{Q}\bigr)$ is the process noise with covariance matrix $\mathbf{Q}$.

Discrete measurements at timestep $k$ are modeled as
$$ \mathbf{z}_k^{camera}
= \begin{bmatrix} x \\ y \\ \theta \end{bmatrix}
= h^{camera} (\mathbf{x}_k) + \mathbf{v}_k^{camera}$$

$$ \mathbf{z}_k^{encoders}
= \begin{bmatrix} v_r \\ v_l \end{bmatrix}
= h^{encoders} (\mathbf{x}_k)+ \mathbf{v}_k^{encoders}$$

where
$\mathbf{v}_k^{camera} \sim \mathcal{N}\bigl(\mathbf{0},\mathbf{R}_k^{camera}\bigr)$ is the camera measurement noise with covariance matrix $\mathbf{R}_k^{camera}$ and
$\mathbf{v}_k^{encoders} \sim \mathcal{N}\bigl(\mathbf{0},\mathbf{R}_k^{encoders}\bigr)$ is the encoders measurement noise with covariance matrix $\mathbf{R}_k^{encoders}$.

The system is linearized around the current estimate using the Jacobian of $f$ :
$$ F(t) = \left. \frac{\partial f}{\partial \mathbf{x}} \right|_{\hat{\mathbf{x}}(t)} = \frac{1}{2} \begin{bmatrix}
0 & 0 & -\sin{\theta} \, (v_r + v_l) & \cos{\theta} & \cos{\theta} \\
0 & 0 & \cos{\theta} \, (v_r + v_l) & \sin{\theta} & \sin{\theta} \\
0 & 0 & 0 & \frac{1}{l} & -\frac{1}{l}  \\
0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0
\end{bmatrix}$$

$$ H_k^{camera}
= \left. \frac{\partial h^{camera}}{\partial \mathbf{x}} \right|_{\hat{\mathbf{x}}_{k|k-1}}
= \begin{bmatrix}
1 & 0 & 0 & 0 & 0 \\
0 & 1 & 0 & 0 & 0 \\
0 & 0 & 1 & 0 & 0
\end{bmatrix}$$

$$ H_k^{encoders}
= \left. \frac{\partial h^{encoders}}{\partial \mathbf{x}} \right|_{\hat{\mathbf{x}}_{k|k-1}}
=  \begin{bmatrix}
0 & 0 & 0 & 1 & 0 \\
0 & 0 & 0 & 0 & 1
\end{bmatrix}$$

The Extended Kalman filter is formulated as a prediction step:
$$
\dot{\hat{\mathbf{x}}}(t) = f(\hat{\mathbf{x}}(t)) \\
\dot{\hat{\mathbf{P}}}(t) = \mathbf{F}(t)\mathbf{P}(t) + \mathbf{P}(t)\mathbf{F}(t)^T + \mathbf{Q}(t)
$$
discretized such that
$$
\hat{\mathbf{x}}_{k} = \hat{\mathbf{x}}_{k-1} + \dot{\hat{\mathbf{x}}}(t) \, dt \\
\hat{\mathbf{P}}_{k} = \hat{\mathbf{P}}_{k-1} + \dot{\hat{\mathbf{P}}}(t) \, dt $$
followed by a prediction step:
$$
\tilde{\mathbf{y}}_{k} = \mathbf{z}_k - h(\hat{\mathbf{x}}_k) \\
\mathbf{S}_k = \mathbf{H}_k \mathbf{P}_k \mathbf{H}_k^T + \mathbf{R}_k \\
\mathbf{K}_k = \mathbf{P}_k \mathbf{H}_k^T + \mathbf{S}_k^{-1} \\
\hat{\mathbf{x}}_k = \hat{\mathbf{x}}_{k-1} + \mathbf{K}_k \tilde{\mathbf{y}}_{k} \\
\mathbf{P}_k = (\mathbf{I} - \mathbf{K}_k \mathbf{H}_K) \mathbf{P}_{k-1}
$$

where $\hat{\mathbf{x}}(t)$ is the state estimate, $\hat{\mathbf{P}}$ is the estimate covariance, $\tilde{\mathbf{y}}_{k}$ is the innovation (or measurement residual), $\mathbf{S}_k$ is the innovation covariance and $\mathbf{K}_k$ is the Near-optimal Kalman gain.

Source : [Extended Kalman Filter](https://en.m.wikipedia.org/wiki/Extended_Kalman_filter#Discrete-time_measurements)

This formulation allows prediction with flexible timesteps.

This filtering is implemented in the file [Motion.py](Motion/Motion.py). Care must be taken to normalize the angle; in our case $\theta \in [-\pi, \pi]$.

In [1]:
# Excerpt from Motion.py

import numpy as np

half_wheelbase = 95/2                        # [mm] distance between wheels divided by 2 (called l in report)

def wrap(angle):
    '''
    Normalize angle in the interval [-pi, pi]
    '''
    return (angle + np.pi) % (2 * np.pi) - np.pi

# --------------------------------------------------------
# Functions used in the dynamic model of the Kalman Filter
def transition_function(x):
    theta = x[2]
    v_r = x[3]
    v_l = x[4]

    x_dot = np.array([
        np.cos(theta)*(v_l+v_r)/2,
        np.sin(theta)*(v_l+v_r)/2,
        (v_r-v_l)/(2*half_wheelbase),
        0,
        0
    ])

    return x_dot

def transition_jacobian(x):
    theta = x[2]
    v_r = x[3]
    v_l = x[4]

    F = np.array([
        [0, 0, -np.sin(theta)*(v_l+v_r)/2, np.cos(theta)/2, np.cos(theta)/2],
        [0, 0, np.cos(theta)*(v_l+v_r)/2, np.sin(theta)/2, np.sin(theta)/2],
        [0, 0, 0, 1/(2*half_wheelbase), -1/(2*half_wheelbase)],
        [0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0]
    ])

    return F

def h_cam(state):
    return state[:3]    # [x, y, theta]

def h_encoder(state):
    return state[3:]    # [vr, vl]


class ExtendedKalmanFilter:
    '''
    Implements an Extended Kalman Filter, assuming no inputs.
    Continous-time model with discrete measurements.

    Assumes: x_dot = f(x) + w(t)        (state evolution)
             z_k = h(x_k) + v_k         (measurement)
    where w is Gaussian white noise with zero mean and covariance Q(t)
          v is Gaussian white noise with zero mean and covariance R_k

    This class stores the last prediction as x_hat and the last estimate as x, along with the estimate covariance P.
    x = [x, y, theta, vr, vl]

    Q and R can be estimated from sensor values but can be tweaked

    References
    ----------
    https://en.wikipedia.org/wiki/Extended_Kalman_filter#Discrete-time_measurements
    '''

    def __init__(self, x0 = np.zeros(5)):
        self.x = x0
        self.x_hat = x0
        self.H_cam = np.hstack((np.eye(3), np.zeros((3,2))))
        self.H_encoder = np.hstack((np.zeros((2,3)), np.eye(2)))
        self.Q = np.diag([10, 10, 0.5, 10, 10])                 # process covariance matrix
        self.R_cam = np.diag([10, 10, 0.1])       # Camera measurement covariance [varX (mm^2), varY(mm^2), varTheta(rad^2)]
        self.R_encoder = np.eye(2)         # Encoder measurement covariance [mm^2/s^2]
        self.P = np.block([                     # initial estimate covariance
            [self.R_cam, np.zeros((self.R_cam.shape[0], self.R_encoder.shape[1]))],
            [np.zeros((self.R_encoder.shape[0], self.R_cam.shape[1])), self.R_encoder]
        ])


    def encoder_update(self, x_hat, P, z):
        return self.update_step(x_hat, P, z, h_encoder,
                                self.H_encoder, self.R_encoder)

    def camera_update(self, x_hat, P, z):
        return self.update_step(x_hat, P, z, h_cam,
                                self.H_cam, self.R_cam)
    


    def predict_step(self, x, P, dt):
        '''
        Return the a priori state estimate and covariance, after the prediction step.

        Parameters
        ----------
            x: previous state
            P: previous estimate covariance
            dt: time since last prediction/update of x
        '''
        

        x_hat_dot = transition_function(x)      # Predicted (a priori) state estimate
        F = transition_jacobian(x)
        P_dot = F@P + P@F.T + self.Q            # Predicted (a priori) estimate covariance
        #P_dot = F(x)@P@F(x).T + Q

        self.x_hat = x + dt*x_hat_dot           # numerical integration
        self.x_hat[2] = wrap(self.x_hat[2])     # normalize angle
        self.P = P + dt*P_dot                   

        return self.x_hat, self.P


    def update_step(self, x_hat, P, z, h, H, R):
        '''
        Return the a posteriori state estimate and covariance, after the update step.

        Parameters
        ----------
            x_hat: previous state estimate (after prediction)
            P: previous estimate covariance
            z: measurement
            h: function, such that z = h(x)
            H: jacobian of h
            R: measurement covariance
        '''
        y_hat = z - h(x_hat)                    # Innovation
        S = H@P@H.T + R                         # Innovation covariance
        K = P@H.T@np.linalg.inv(S)              # Optimal Kalman gain
        self.x = x_hat + K@y_hat                # Updated (a posteriori) state estimate
        self.x[2] = wrap(self.x[2])             # normalize angle
        self.P = (np.eye(H.shape[1]) - K@H)@P   # Updated (a posteriori) estimate covariance

        return self.x, self.P


### Sensor noise covariance measurement

To estimate the encoder covariance noise, we can collect data samples at steady state.

In [ ]:
from tdmclient import ClientAsync
import numpy as np
import matplotlib.pyplot as plt

n_samples = 500
wait_time = 1e-2    # motor.speed sensor refresh period (cf. Cheat Sheet)
target_speed = 50

data = []
 
with ClientAsync() as client:
    with await client.lock() as node:
        
        await node.set_variables({
            "motor.left.target": [target_speed],
            "motor.right.target": [target_speed]
            })
                
        for i in range(n_samples):
            await client.sleep(wait_time)
            client.aw(node.wait_for_variables())
            data.append(({"ground_sensor1":node["prox.ground.reflected"][0],
                          "ground_sensor2":node["prox.ground.reflected"][1],  
                            "left_speed":node["motor.left.speed"],
                            "right_speed":node["motor.right.speed"]}))

        await node.set_variables({
            "motor.left.target": [0],
            "motor.right.target": [0]
            })

        data = {'name': node.props["name"], 'target_speed': target_speed, 'data': data}


left_speed = np.array([sample["left_speed"] for sample in data])
right_speed = np.array([sample["right_speed"] for sample in data])

plt.plot(left_speed)
plt.plot(right_speed)
plt.axhline(target_speed)
plt.show()


# Cut transient acceleration
right_speed = right_speed[200:]
left_speed = left_speed[200:]

speed_to_mms = 0.42         

avg_speed = (left_speed+right_speed)/2
std_speed = np.std(avg_speed/speed_to_mms)  # [mm^2/s^2]
cov_speed = np.cov(np.vstack((right_speed, left_speed))/speed_to_mms)

print(cov_speed)        # Linked to R_encoder [mm^2/s^2]

We find (in $mm^2/s^2$) $$R \approx \begin{bmatrix}
2.3  & 0.75 \\
0.75 &  1.9
\end{bmatrix}$$
Some of this variance might be due to other factors (e.g. closed-loop control, surface grip) rather than the sensors itself. Therefore, we tweak $R^{encoder} \approx \mathbf{I}$ and will increase the process covariance matrix $\mathbf{Q}$ to account for this noise. 

The camera localization has very low variance in steady state. However we can estimate that, with our setup, its resolution is approximately of 1 cm and the angle detection resolution might be close to 0.1 radians. Therefore, we set 
$$ R^{camera} \approx \begin{bmatrix}
10 & 0 & 0 \\
0 & 10 & 0 \\
0 & 0 & 0.1
\end{bmatrix}$$
where the units along the diagonal are $[mm^2/s^2, mm^2/s^2, rad^2]$.

## Control law: Move to Pose

Define
$$
\rho = \sqrt{\Delta x^2 + \Delta y^2} \\
\alpha = -\theta + \operatorname{atan2} \frac{\Delta y}{\Delta x} \\
\beta = -\theta - \alpha
$$
where $\Delta x$ and $\Delta y$ are the distances along each axis between the current position of the robot and the goal.
Note that this coordinate transformation is not defined at $x = y = 0$, therefore the controller must be stopped close to the goal.

It can be shown that the proportional controller $v = k_\rho \rho$, $w = k_\alpha \alpha + k_\beta \beta$ will drive the robot to $(\rho, \alpha, \beta) = (0,0,0)$. The corresponding wheel speed outputs are then
$$ \begin{bmatrix}
v_r \\
v_l
\end{bmatrix} = \begin{bmatrix}
1 & l \\
1 & -l
\end{bmatrix} \begin{bmatrix}
v \\
w
\end{bmatrix}
$$

$k_\rho$ controls how fast the robot will drive towards its reference, $k_\alpha$ controls how fast it turns toward the reference and $k_\beta$ controls how fast it aligns itself with the reference angle. Exact values have been determined by trial and error; they can be determined independently.

Source: Autonomous Mobile Robots @ ETHZ, [Exercice 2: Mobile Robot Kinematics and Control](https://ethz.ch/content/dam/ethz/special-interest/mavt/robotics-n-intelligent-systems/asl-dam/documents/lectures/autonomous_mobile_robots/spring-2020/Exercise2/amr_exercise_2_slides.pdf)

This controller is implemented in the file [Motion.py](Motion/Motion.py). The tangential speed of the robot is saturated, and the angle error is normalized.

In [ ]:
# Excerpt from Motion.py

# ---------------------
# Move to Pose control law
## Controller parameters
## Stable if : Krho > 0, Kbeta < 0, Kalpha > Krho
Krho = 1
Kalpha = 3
Kbeta = -0.5
K = np.array([Krho, Kalpha, Kbeta])
MAX_SPEED = 200     # maximum tangential speed


def direction_to_goal(position, goal):
    '''
    Assumes that both position and goals are array formatted as
    [x, y, angle]

    Returns
    -------
    rho: distance to goal
    alpha : angle to goal
    beta : (theta_ref - theta - alpha), to align with goal orientation
    '''
    dx = goal[0] - position[0]
    dy = goal[1] - position[1]

    rho = np.linalg.norm([dx, dy])
    alpha = wrap(np.arctan2(dy, dx) - position[2])
    beta = wrap(goal[2] - position[2] - alpha)

    return rho, alpha, beta


def control_law(rho, alpha, beta):
    '''
    Parameters
    ----------
    rho: distance to goal
    alpha : angle to goal
    beta : (theta_ref - theta - alpha), to align with goal orientation
    K = [Krho, Kalpha, Kbeta] : controller parameters        
    '''

    v = K[0]*rho
    if v > MAX_SPEED:
        v = MAX_SPEED

    w = K[1]*alpha + K[2]*beta

    v_r = v + half_wheelbase*w
    v_l = v - half_wheelbase*w

    return [v_r, v_l]


## Local navigation

As a final functionality of the robot's trajectory, the robot needs to be able to avoid obstacles that the camera has not detected, or avoid those obstacles in the case of camera obstruction, while at the same time preventing a collision with the original obstacles detected, now called "global obstacles". As explained in the environment section, these global obstacles are represented in green, and are 2D shapes. On the other hand, the so-called "local obstacles" are represented in our environment as 3D obstacles and seen from above as black shapes. These black shapes are filtered out by the masks applied on the camera's frames, which means that they are not at any point actually detected by the camera.They can only be detected by the 6 horizontal proximity sensors provided in the Thymio, 4 in front, and 2 in the back, as seen in the diagram of Thymio sensors below. The proximity sensors work by sending a beam of infrared light and analyzing the reflected beam it receives from obstacles directly in front of them. Objects situated close to the sensor reflect a stronger light signal compared to those located further away from it.

Source: Micro-452 Basics of Mobile Robotics course at EPFL: Solutions Sesson 4 - Local Navigation.ipynb

#### Diagram of Thymio Sensors

<img src="Figures/Local_Nav/Thymio_Sensors.png" width="600">

Source: [Thymio Cheat Sheet](https://moodle.epfl.ch/pluginfile.php/2706097/mod_resource/content/1/ThymioCheatSheet.pdf)

The robot thus updates its motor speeds continuously according to new information. If the proximity sensors detect an obstacle, the new motor speeds are then influenced by this information. However, the local avoidance of obstacles could lead the robot to failure if it chooses instead to run into global obstacles in an attempt to avoid the local ones. This is why we chose to incorporate a virtual sensor as well in order to avoid this possibility. The virtual sensor needs to be able to detect the green 2D shapes in the environment. We thus artificially created a sensor that simulates the function of the proximity sensors already in the robot by scanning for obstacles in 5 places at the front of the robot, like the 5 actual horizontal proximity sensors. However, instead of scanning for obstacles up until a certain distance, it takes a single point that's a specific distance away, namely, half the size of the robot, and reads the filtered image provided by the camera to see if the point lies within a global obstacle. The red points demonstrate the presence of the global obstacle, and the green ones demonstrate the lack of obstacle for those 5 pixels chosen. 

#### Virtual Sensor Demonstration

<img src="Figures/Local_Nav/Virtual Sensor Simulation.png" width="300">

This virtual sensor is only called if a local obstacle has been detected, since the robot's original motor speeds were already decided using the avoidance of global obstacles as a main deciding factor. The only situation in which the robot might run into a global obstacle is if a local obstacle obstructs the planned path. 

### Artificial Neural Network Algorithm

The local navigation portion of this project comes at the very end of the main function because the robot makes its decisions on its motor speeds mostly according to a planned trajectory it calculates at the beginning, with continuous updates from its current estimated state. The local navigation function works as a reactionary safe-guard measure at the end of the decision-making process by taking as argument the speed of the right and left motors previously calculated, and overriding those values in the case of local or global obstacle detection. This is done using an artificial feedforward neural network that uses the current motor speeds as inputs, as well as the horizontal proximity sensor and virtual sensor data, and outputs the new motor speeds. The new motor speeds are identical to the previous motor speeds if there are no obstacles detected by the proximity sensors. 

Our artificial feedforward neural network works only in one direction, and is called at every loop of the main function to update the motor speeds according to new information. It is composed simply of an input layer and an output layer. The input layer has 14 nodes: each of the 7 horizontal proximity sensor data points, each of the 5 virtual sensor data points, and the 2 motor speeds previously calculated. The output layer only has 2 nodes: the right and left motor speeds. Each of the nodes has a weight attached to them to be able to regulate how each of them influences the motor speeds. The visualization of how the horizontal proximity sensors influences each motor speed can be seen in the ANN Robot Control image shown below. 

#### ANN Robot Control for Horizontal Proximity Sensors

<img src="Figures/Local_Nav/ANN_robot_control copy.png" width="300">

Source: Micro-452 Basics of Mobile Robotics course at EPFL: Solutions Sesson 3 - Artificial neural networks.ipynb

Each node's value is multiplied by its specific weight and added to the current motor speeds to finally output the new motor speeds. If the proximity sensors data is 0, meaning there are no obstacles to avoid, the current motor speed remains unaffected. A sample of the algorithm is provided below.

In [ ]:
import numpy as np
from Local_Navigation.LocalNav import *
sensor_scale = 100

w_l_virt = np.array([-3,  -2, 1, 3, 4])*8
w_r_virt = -w_l_virt

w_l_prox = np.array([-3,  -2, 1, 3, 4,  0, 0])*8
w_r_prox = -w_l_prox

# This is the function called in the main loop, which takes the current estimated state of the robot
# and an image of its environment and returns the updated motor speeds and the points to be drawn on the video
def LocalNavigation(prox_horizontal, image, robot_size, robot_position, robot_angle, robot_speed):
    
    x_virt, points = get_VirtualSensor(image, robot_position, robot_angle, robot_size)
    y = avoid_obstacles(prox_horizontal, x_virt, robot_speed)

    return y, points, x_virt


# This is the main ANN algorithm, taking the current speed and horizontal & virtual proximity sensor data
# and outputs the updated speed
def avoid_obstacles(prox_horizontal, x_virt, robot_speed):
    
    y = [robot_speed[0], robot_speed[1]]
    x_prox = [0,0,0,0,0,0,0]

    if np.all(prox_horizontal == 0):
        return y

    for i in range(len(x_prox)):
        # Get and scale proximity inputs
        x_prox[i] = prox_horizontal[i] // sensor_scale

        # Compute outputs of prox neurons and set motor powers
        y[0] = y[0] + x_prox[i] * w_l_prox[i]
        y[1] = y[1] + x_prox[i] * w_r_prox[i]

    for i in range(len(x_virt)):
        # Compute outputs of virtual neurons and set motor powers
        y[0] = y[0] + x_virt[i] * w_l_virt[i]
        y[1] = y[1] + x_virt[i] * w_r_virt[i]

    # If robot is stuck going backwards, choose a direction to turn in and reevaluate in next loop
    if (y[0] < 0 and y[1] < 0):
        y[1] = -y[1]

    return y 

## Putting it all together: the main.py file

![A Diagram of the structure of the Miniproject](Figures/Diagram.png)


## Running the overall project

In [ ]:
!python main.py